# 🧪 How2Sign — Inference, ONNX Export & Camera Testing

**Loads the trained CSLR model from the training notebook and runs:**
1. Full test-set evaluation — WER, CER, MER, SER, WIL, WIP, BLEU-1, Precision, Recall, F1
2. ONNX export (SavedModel strategy + INT8 quantization fallback)
3. ONNX vs TF numerical validation
4. Real-time webcam / video inference via MediaPipe (232-dim with face landmarks)

**Requires:** `best_cslr.weights.h5`, `vocab.json`, `model_config.json` from the training notebook.

In [1]:
# ============================================================
# 1. SETUP & LOAD SAVED ARTIFACTS
# ============================================================
import os, json, math, time
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from collections import deque, Counter

try:
    import h5py
except ImportError:
    os.system('pip install -q h5py')
    import h5py

# --- GPU setup -----------------------------------------------------------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')

print('=' * 55)
print(f'  TensorFlow : {tf.__version__}')
print(f'  NumPy      : {np.__version__}')
if gpus:
    for i, g in enumerate(gpus):
        d = tf.config.experimental.get_device_details(g)
        print(f'  GPU {i}      : {d.get("device_name", g.name)}')
else:
    print('  Mode       : CPU only')
print('=' * 55)

# --- Paths ---------------------------------------------------------------
ARTIFACTS_DIR = Path('/kaggle/working')       # training notebook output
BASE_DIR      = Path('/kaggle/input/datasets/hadeelgamal/artifacts2')

VOCAB_PATH   = ARTIFACTS_DIR / 'vocab.json'
WEIGHTS_PATH = ARTIFACTS_DIR / 'best_cslr.weights.h5'
CONFIG_PATH  = ARTIFACTS_DIR / 'model_config.json'
CSV_TEST     = BASE_DIR / 'how2sign_test_subset.csv'
FEAT_DIR_TEST= BASE_DIR / 'test_features'
OUTPUT_DIR   = ARTIFACTS_DIR
ONNX_PATH    = OUTPUT_DIR / 'cslr_inference.onnx'

for label, p in [('VOCAB',    VOCAB_PATH),  ('WEIGHTS', WEIGHTS_PATH),
                  ('CONFIG',   CONFIG_PATH), ('CSV_TEST', CSV_TEST)]:
    print(f'  {label:8s} exists={p.exists()}  {p}')

# --- Load vocab & model config ------------------------------------------
with open(VOCAB_PATH) as f:
    vocab = json.load(f)
with open(CONFIG_PATH) as f:
    cfg = json.load(f)

SEQUENCE_LENGTH = cfg['seq_len']          # e.g. 424
NUM_FEATURES    = cfg['num_features']     # 232
MAX_LABEL       = cfg['max_label']        # 50
VOCAB_SIZE_CTC  = cfg['vocab_size_ctc']   # MAX_TOKENS + 1
MAX_TOKENS      = cfg['max_tokens']       # 3000

print(f'\nModel: seq={SEQUENCE_LENGTH}  feats={NUM_FEATURES}  '
      f'vocab_ctc={VOCAB_SIZE_CTC}  '
      f'best_epoch={cfg["best_epoch"]}  '
      f'best_val_loss={cfg["best_val_loss"]:.4f}')

# --- Decode helpers ------------------------------------------------------
def decode_indices(indices) -> str:
    words = []
    for idx in indices:
        if idx == 0:
            continue
        w_idx = int(idx) - 1
        if 0 <= w_idx < len(vocab):
            w = vocab[w_idx]
            if w not in ('', '[UNK]'):
                words.append(w)
    return ' '.join(words)

def decode_logits(logits: np.ndarray, input_len: int) -> str:
    decoded, _ = tf.keras.backend.ctc_decode(
        logits[np.newaxis], input_length=np.array([input_len]), greedy=True)
    return decode_indices(decoded[0][0].numpy())


Your GPU may run slowly with dtype policy mixed_float16 because it does not have compute capability of at least 7.0. Your GPU:
  NVIDIA GeForce MX150, compute capability 6.1
See https://developer.nvidia.com/cuda-gpus for a list of GPUs and their compute capabilities.
If you will use compatible GPU(s) not attached to this host, e.g. by running a multi-worker model, you can ignore this warning. This message will only be logged once
  TensorFlow : 2.10.0
  NumPy      : 1.23.5
  GPU 0      : NVIDIA GeForce MX150
  VOCAB    exists=False  \kaggle\working\vocab.json
  WEIGHTS  exists=False  \kaggle\working\best_cslr.weights.h5
  CONFIG   exists=False  \kaggle\working\model_config.json
  CSV_TEST exists=False  \kaggle\input\datasets\hadeelgamal\artifacts2\how2sign_test_subset.csv


FileNotFoundError: [Errno 2] No such file or directory: '\\kaggle\\working\\vocab.json'

## 2. Rebuild Model & Load Weights

In [ ]:
# ============================================================
# 2. REBUILD MODEL ARCHITECTURE & LOAD WEIGHTS
# ============================================================
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization,
    LSTM, Bidirectional, Softmax,
    MultiHeadAttention, LayerNormalization
)
from tensorflow.keras.models import Model


class CTCLossLayer(tf.keras.layers.Layer):
    """tf.nn.ctc_loss — Keras 3 compatible, blank_index=0."""
    def __init__(self, blank_index=0, **kwargs):
        super().__init__(**kwargs)
        self.blank_index = blank_index

    def call(self, inputs):
        y_pred, labels, input_length, label_length = inputs
        log_probs    = tf.math.log(y_pred + 1e-8)
        log_probs_tm = tf.transpose(log_probs, [1, 0, 2])
        input_len_1d = tf.cast(tf.reshape(input_length, [-1]), tf.int32)
        label_len_1d = tf.cast(tf.reshape(label_length, [-1]), tf.int32)
        label_len_1d = tf.minimum(label_len_1d, input_len_1d)
        label_len_1d = tf.maximum(label_len_1d, 1)
        loss = tf.nn.ctc_loss(
            labels=labels, logits=log_probs_tm,
            label_length=label_len_1d, logit_length=input_len_1d,
            logits_time_major=True, blank_index=self.blank_index)
        loss = tf.where(tf.math.is_finite(loss), loss, tf.zeros_like(loss))
        return tf.reduce_mean(loss)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'blank_index': self.blank_index})
        return cfg


def build_model(seq_len, num_features, vocab_size_ctc, max_label):
    inp          = Input(shape=(seq_len, num_features), name='input')
    labels_in    = Input(shape=(None,), dtype='int32',  name='labels')
    input_len_in = Input(shape=(1,),    dtype='int32',  name='input_length')
    label_len_in = Input(shape=(1,),    dtype='int32',  name='label_length')

    x1   = Bidirectional(LSTM(256, return_sequences=True), name='bilstm_1')(inp)
    x1   = BatchNormalization(name='bn_1')(x1)
    attn = MultiHeadAttention(num_heads=4, key_dim=64, name='mha')(x1, x1)
    x    = LayerNormalization(name='ln_1')(x1 + attn)
    x    = Dropout(0.3, name='drop_1')(x)
    x    = Bidirectional(LSTM(256, return_sequences=True), name='bilstm_2')(x)
    x    = BatchNormalization(name='bn_2')(x)
    x    = Dropout(0.3, name='drop_2')(x)
    logits = Dense(vocab_size_ctc, name='logits')(x)
    y_pred = Softmax(name='prediction')(logits)

    ctc_out = CTCLossLayer(blank_index=0, name='ctc_loss')(
        [y_pred, labels_in, input_len_in, label_len_in])

    model_train = Model(
        inputs=[inp, labels_in, input_len_in, label_len_in],
        outputs=ctc_out, name='ctc_train')
    model_train.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=5.0))
    model_infer = Model(inputs=inp, outputs=y_pred, name='ctc_inference')
    return model_train, model_infer


model_train, model_infer = build_model(
    SEQUENCE_LENGTH, NUM_FEATURES, VOCAB_SIZE_CTC, MAX_LABEL)

if WEIGHTS_PATH.exists():
    model_train.load_weights(str(WEIGHTS_PATH))
    print(f'Weights loaded from {WEIGHTS_PATH}')
else:
    print('WARNING: weights file not found — model has random weights.')

print(f'Inference model  input : {model_infer.input_shape}')
print(f'Inference model output : {model_infer.output_shape}')


## 3. Full Evaluation on Test Set

| Metric | Meaning | Direction |
|---|---|---|
| WER | Word Error Rate | lower ↓ |
| CER | Character Error Rate | lower ↓ |
| MER | Match Error Rate | lower ↓ |
| SER | Sentence Error Rate | lower ↓ |
| WIL | Word Information Lost | lower ↓ |
| WIP | Word Information Preserved | higher ↑ |
| BLEU-1 | Unigram translation quality | higher ↑ |
| Precision / Recall / F1 | Token-level micro-averaged | higher ↑ |

In [ ]:
!pip install -q jiwer scikit-learn


In [ ]:
# ============================================================
# 3. EVALUATION & METRICS
# ============================================================
import jiwer
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import precision_score, recall_score, f1_score


class TestGenerator(tf.keras.utils.Sequence):
    def __init__(self, csv_path, feat_dir, batch_size=32):
        if csv_path.exists():
            df = pd.read_csv(csv_path, on_bad_lines='skip')
            if 'NPY_PATH' in df.columns:
                df = df[df['NPY_PATH'].notna()].reset_index(drop=True)
            self.df = df
        else:
            self.df = pd.DataFrame()
        self.feat_dir = feat_dir
        self.bs = batch_size

    def __len__(self):
        return max(1, int(np.ceil(len(self.df) / self.bs)))

    def __getitem__(self, idx):
        batch = self.df.iloc[idx * self.bs : (idx + 1) * self.bs]
        X     = np.zeros((len(batch), SEQUENCE_LENGTH, NUM_FEATURES), dtype=np.float32)
        lens  = np.zeros(len(batch), dtype=np.int32)
        sents = []
        for i, (_, row) in enumerate(batch.iterrows()):
            npy_path = BASE_DIR / str(row.get('NPY_PATH', ''))
            if npy_path.exists():
                frames = np.load(npy_path)
                T = min(len(frames), SEQUENCE_LENGTH)
                X[i, :T, :] = frames[:T]
                lens[i] = T
            sents.append(str(row.get('SENTENCE', '')))
        return X, lens, sents


def evaluate_model(model_inf, csv_path, feat_dir, max_batches=None):
    gen = TestGenerator(csv_path, feat_dir)
    if len(gen.df) == 0:
        print('No test data found.')
        return [], []

    n_batches = min(max_batches or len(gen), len(gen))
    all_refs, all_hyps = [], []

    print(f'Inference on {n_batches} batches…')
    for i in range(n_batches):
        X, lens, refs = gen[i]
        logits = model_inf.predict(X, verbose=0)          # (B, T, V+1)
        decoded, _ = tf.keras.backend.ctc_decode(
            logits, input_length=lens, greedy=True)
        preds = decoded[0].numpy()
        for j in range(len(refs)):
            all_hyps.append(decode_indices([x for x in preds[j] if x != -1]) or '<empty>')
            all_refs.append(refs[j].strip() or '<empty>')

    # --- jiwer transforms -----------------------------------------------
    W = jiwer.Compose([
        jiwer.ToLowerCase(), jiwer.RemovePunctuation(),
        jiwer.RemoveMultipleSpaces(), jiwer.Strip(),
        jiwer.ReduceToListOfListOfWords(),
    ])
    C = jiwer.Compose([
        jiwer.ToLowerCase(), jiwer.RemovePunctuation(),
        jiwer.ReduceToListOfListOfChars(),
    ])

    wer = jiwer.wer(all_refs, all_hyps, reference_transform=W, hypothesis_transform=W)
    mer = jiwer.mer(all_refs, all_hyps, reference_transform=W, hypothesis_transform=W)
    wil = jiwer.wil(all_refs, all_hyps, reference_transform=W, hypothesis_transform=W)
    wip = jiwer.wip(all_refs, all_hyps, reference_transform=W, hypothesis_transform=W)
    cer = jiwer.cer(all_refs, all_hyps, reference_transform=C, hypothesis_transform=C)
    ser = sum(r.strip() != h.strip() for r, h in zip(all_refs, all_hyps)) / max(len(all_refs), 1)

    def bleu1(refs, hyps):
        match = total = 0
        for r, h in zip(refs, hyps):
            rc = Counter(r.split())
            for w in h.split():
                if rc.get(w, 0) > 0:
                    match += 1; rc[w] -= 1
                total += 1
        return match / max(total, 1)

    bleu = bleu1(all_refs, all_hyps)

    ref_w = [w for s in all_refs for w in s.split()]
    hyp_w = [w for s in all_hyps for w in s.split()]
    mx    = max(len(ref_w), len(hyp_w))
    ref_w += ['<pad>'] * (mx - len(ref_w))
    hyp_w += ['<pad>'] * (mx - len(hyp_w))
    all_t = sorted(set(ref_w + hyp_w)); t2i = {t: i for i, t in enumerate(all_t)}
    r_ids = [t2i[w] for w in ref_w];   h_ids = [t2i[w] for w in hyp_w]
    prec  = precision_score(r_ids, h_ids, average='micro', zero_division=0)
    rec   = recall_score   (r_ids, h_ids, average='micro', zero_division=0)
    f1    = f1_score       (r_ids, h_ids, average='micro', zero_division=0)

    # --- Print table ----------------------------------------------------
    n = len(all_refs)
    print(f'\n{"="*55}')
    print(f'  Evaluation on {n} sentences')
    print(f'{"="*55}')
    print(f'  WER  (Word Error Rate)       : {wer:.4f}  ({wer*100:.1f}%)')
    print(f'  CER  (Char Error Rate)       : {cer:.4f}  ({cer*100:.1f}%)')
    print(f'  MER  (Match Error Rate)      : {mer:.4f}  ({mer*100:.1f}%)')
    print(f'  SER  (Sentence Error Rate)   : {ser:.4f}  ({ser*100:.1f}%)')
    print(f'  WIL  (Word Info Lost)        : {wil:.4f}')
    print(f'  WIP  (Word Info Preserved)   : {wip:.4f}')
    print(f'  BLEU-1                       : {bleu:.4f}')
    print(f'{"-"*55}')
    print(f'  Token Precision (micro)      : {prec:.4f}')
    print(f'  Token Recall    (micro)      : {rec:.4f}')
    print(f'  Token F1        (micro)      : {f1:.4f}')
    print(f'{"="*55}')

    print('\nSample predictions (first 5):')
    for i in range(min(5, n)):
        print(f'  REF : {all_refs[i]}')
        print(f'  HYP : {all_hyps[i]}')
        print()

    # --- Bar chart ------------------------------------------------------
    labels = ['WER','CER','MER','SER','WIL','1-WIP','1-BLEU','Prec','Rec','F1']
    values = [wer, cer, mer, ser, wil, 1-wip, 1-bleu, prec, rec, f1]
    colors = ['#e74c3c']*7 + ['#2ecc71']*3
    fig, ax = plt.subplots(figsize=(13, 4))
    bars = ax.bar(labels, values, color=colors, edgecolor='white', linewidth=0.5)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)
    ax.set_ylim(0, 1.2); ax.set_ylabel('Score')
    ax.set_title('CSLR Test-Set Evaluation Metrics')
    red_p   = mpatches.Patch(color='#e74c3c', label='Error metrics (lower is better)')
    green_p = mpatches.Patch(color='#2ecc71', label='Quality metrics (higher is better)')
    ax.legend(handles=[red_p, green_p], fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'evaluation_metrics.png'), dpi=150)
    plt.show()
    return all_refs, all_hyps


refs, hyps = evaluate_model(model_infer, CSV_TEST, FEAT_DIR_TEST)


## 4. ONNX Export

Uses **SavedModel → tf2onnx CLI** as the primary strategy (most stable for BiLSTM).  
Falls back to Keras direct conversion if SavedModel fails.  
Optionally quantizes to **INT8** (~2× faster CPU inference, ~4× smaller model).

In [ ]:
!pip install -q tf2onnx onnx onnxruntime


In [ ]:
# ============================================================
# 4. ONNX EXPORT  (SavedModel strategy + INT8 fallback)
# ============================================================
import tf2onnx, onnx

SAVED_MODEL_DIR = str(OUTPUT_DIR / 'cslr_saved_model')
ONNX_FP32_PATH  = str(OUTPUT_DIR / 'cslr_inference.onnx')
ONNX_INT8_PATH  = str(OUTPUT_DIR / 'cslr_int8.onnx')

# --- Strategy 1: SavedModel → tf2onnx CLI (most stable for BiLSTM) -----
try:
    model_infer.save(SAVED_MODEL_DIR)
    print(f'SavedModel written to {SAVED_MODEL_DIR}')
    ret = os.system(
        f'python -m tf2onnx.convert '
        f'--saved-model {SAVED_MODEL_DIR} '
        f'--output {ONNX_FP32_PATH} '
        f'--opset 13'
    )
    if ret == 0:
        print(f'ONNX (FP32) saved -> {ONNX_FP32_PATH}  '
              f'({os.path.getsize(ONNX_FP32_PATH)/1e6:.1f} MB)')
    else:
        raise RuntimeError('tf2onnx CLI returned non-zero exit code')
except Exception as e1:
    print(f'SavedModel strategy failed: {e1}')
    print('Falling back to Keras direct conversion…')
    try:
        input_sig = [tf.TensorSpec([1, SEQUENCE_LENGTH, NUM_FEATURES],
                                   tf.float32, name='input')]
        onnx_model, _ = tf2onnx.convert.from_keras(
            model_infer, input_signature=input_sig, opset=13)
        with open(ONNX_FP32_PATH, 'wb') as f:
            f.write(onnx_model.SerializeToString())
        print(f'ONNX (FP32) saved via Keras direct -> {ONNX_FP32_PATH}')
    except Exception as e2:
        print(f'Both strategies failed: {e2}')

# --- Validate ONNX graph -----------------------------------------------
if os.path.exists(ONNX_FP32_PATH):
    onnx.checker.check_model(ONNX_FP32_PATH)
    print('ONNX graph validation: PASSED')

# --- Strategy 2: INT8 dynamic quantization (~2x faster CPU) ------------
try:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    quantize_dynamic(ONNX_FP32_PATH, ONNX_INT8_PATH,
                     weight_type=QuantType.QInt8)
    print(f'ONNX (INT8) saved -> {ONNX_INT8_PATH}  '
          f'({os.path.getsize(ONNX_INT8_PATH)/1e6:.1f} MB)')
except Exception as e:
    print(f'INT8 quantization skipped: {e}')


In [ ]:
# ============================================================
# 4b. ONNX vs TF NUMERICAL VALIDATION
# ============================================================
import onnxruntime as ort

sess = ort.InferenceSession(
    ONNX_FP32_PATH,
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])

dummy    = np.random.rand(1, SEQUENCE_LENGTH, NUM_FEATURES).astype(np.float32)
tf_out   = model_infer.predict(dummy, verbose=0)[0]        # (T, V+1)
ort_out  = sess.run(None, {'input': dummy})[0][0]          # (T, V+1)

max_diff  = float(np.abs(tf_out - ort_out).max())
mean_diff = float(np.abs(tf_out - ort_out).mean())

print(f'TF  output shape  : {tf_out.shape}')
print(f'ORT output shape  : {ort_out.shape}')
print(f'Max  abs diff     : {max_diff:.6f}')
print(f'Mean abs diff     : {mean_diff:.6f}')
print(f'Numerical check   : {"PASSED" if max_diff < 1e-3 else "WARNING > 1e-3"}')

tf_text  = decode_logits(tf_out,  SEQUENCE_LENGTH)
ort_text = decode_logits(ort_out, SEQUENCE_LENGTH)
print(f'\nTF  decoded  : "{tf_text}"')
print(f'ORT decoded  : "{ort_text}"')
print(f'Text match   : {tf_text == ort_text}')


## 5. Real-Time Webcam / Video Inference

**Features:** 232-dim — same layout as training: body (50) + left-hand (42) + right-hand (42) + **face (98)**

Face keypoints capture eyebrow raises, eye shape, and mouth movements — all grammatically meaningful in ASL.

| Key | Action |
|---|---|
| `q` | Quit |
| `r` | Reset sentence |
| `SPACE` | Pause / resume |

```python
run_inference(source=0)              # webcam
run_inference(source='video.mp4')    # video file
run_inference(source=0, use_onnx=True)  # ONNX Runtime backend
```

In [ ]:
!pip install -q mediapipe==0.10.14 opencv-python-headless


In [ ]:
# ============================================================
# 5a. MEDIAPIPE → 232-DIM FEATURE EXTRACTOR
# ============================================================
# Face keypoint indices used in training (49 keypoints × 2 = 98 dims)
_FACE_KP_INDICES = (
    [17,18,19,20,21] + [22,23,24,25,26] +          # eyebrows L+R (10)
    [36,37,38,39,40,41] + [42,43,44,45,46,47] +    # eyes L+R (12)
    [68,69] +                                       # pupils (2)
    [27,28,29,30] + [33] +                          # nose bridge + tip (5)
    [48,49,50,51,52,53,54,55,56,57,58,59] +         # outer lips (12)
    [60,61,62,63,64,65,66,67]                        # inner lips (8)
)  # 49 total

# MediaPipe Face Mesh (468 pts) → OpenPose-70 approximate index mapping
_MP_FACE_MAP = {
    17:70, 18:63, 19:105, 20:66, 21:107,
    22:336,23:296,24:334,25:293,26:300,
    36:33, 37:160,38:158,39:133,40:153,41:144,
    42:362,43:385,44:387,45:263,46:373,47:380,
    68:468,69:473,
    27:6,  28:197,29:195,30:5,
    33:1,
    48:61, 49:185,50:40, 51:39, 52:37, 53:0,
    54:267,55:269,56:270,57:409,58:291,59:375,
    60:78, 61:191,62:80, 63:81,
    64:311,65:310,66:415,67:308,
}


def mediapipe_to_232dim(results) -> np.ndarray:
    """
    Convert MediaPipe Holistic result → (232,) float32.
    Layout: body(50) + left_hand(42) + right_hand(42) + face(98) = 232
    """
    # ── Body: 25 OpenPose joints × (x,y) = 50 ──────────────────────────
    pose = np.zeros((25, 2), dtype=np.float32)
    if results.pose_landmarks:
        lm = results.pose_landmarks.landmark
        def sp(op, mp):
            if mp < len(lm):
                pose[op] = [lm[mp].x, lm[mp].y]
        sp(0,0); sp(2,12); sp(3,14); sp(4,16)
        sp(5,11);sp(6,13); sp(7,15); sp(9,24)
        sp(10,26);sp(11,28);sp(12,23);sp(13,25)
        sp(14,27);sp(15,5); sp(16,2); sp(17,8)
        sp(18,7); sp(19,31);sp(21,29);sp(22,32);sp(24,30)
        # Interpolate Neck (1) and MidHip (8)
        if pose[2].any() and pose[5].any():
            pose[1] = (pose[2] + pose[5]) / 2.0
        if pose[9].any() and pose[12].any():
            pose[8] = (pose[9] + pose[12]) / 2.0

    # ── Hands: 21 × (x,y) each = 42+42 ─────────────────────────────────
    lh = np.zeros((21, 2), dtype=np.float32)
    if results.left_hand_landmarks:
        for k, pt in enumerate(results.left_hand_landmarks.landmark):
            lh[k] = [pt.x, pt.y]

    rh = np.zeros((21, 2), dtype=np.float32)
    if results.right_hand_landmarks:
        for k, pt in enumerate(results.right_hand_landmarks.landmark):
            rh[k] = [pt.x, pt.y]

    # ── Face: 49 selected keypoints × (x,y) = 98 ────────────────────────
    face = np.zeros((49, 2), dtype=np.float32)
    if results.face_landmarks:
        mesh = results.face_landmarks.landmark
        for k, op_idx in enumerate(_FACE_KP_INDICES):
            mp_idx = _MP_FACE_MAP.get(op_idx)
            if mp_idx is not None and mp_idx < len(mesh):
                face[k] = [mesh[mp_idx].x, mesh[mp_idx].y]

    return np.concatenate([
        pose.flatten(),   # 50
        lh.flatten(),     # 42
        rh.flatten(),     # 42
        face.flatten(),   # 98
    ]).astype(np.float32)  # 232 total


print('232-dim feature extractor ready.')
print('Feature regions: body[0:50] lhand[50:92] rhand[92:134] face[134:232]')


In [ ]:
# ============================================================
# 5b. STABILIZATION TRACKER
# ============================================================
class StabilizationTracker:
    """
    Sliding-window majority-vote smoother.
    Commits a word only when it wins >= majority_ratio of the window
    AND a cooldown period has elapsed since the last commit.
    """
    def __init__(self, window_size=15, majority_ratio=0.6, cooldown_s=1.0):
        self.window_size    = window_size
        self.majority_ratio = majority_ratio
        self.cooldown_s     = cooldown_s
        self.buffer         = deque(maxlen=window_size)
        self.last_time      = 0.0
        self.last_word      = ''
        self.sentence       = []

    def update(self, predicted_word: str):
        self.buffer.append(predicted_word or None)
        if len(self.buffer) < self.window_size:
            return None
        counts = Counter(w for w in self.buffer if w)
        if not counts:
            return None
        top_word  = counts.most_common(1)[0][0]
        top_ratio = counts[top_word] / self.window_size
        now = time.time()
        if (top_ratio >= self.majority_ratio
                and top_word != self.last_word
                and (now - self.last_time) > self.cooldown_s):
            self.sentence.append(top_word)
            self.last_word = top_word
            self.last_time = now
            self.buffer.clear()
            return top_word
        return None

    def get_sentence(self, last_n=8) -> str:
        return ' '.join(self.sentence[-last_n:])

    def reset(self):
        self.buffer.clear(); self.sentence.clear()
        self.last_word = ''; self.last_time = 0.0


print('StabilizationTracker ready.')


In [ ]:
# ============================================================
# 5c. WEBCAM / VIDEO INFERENCE LOOP  (232-dim + face)
# ============================================================
import cv2
import mediapipe as mp

mp_holistic = mp.solutions.holistic
mp_drawing  = mp.solutions.drawing_utils
mp_draw_sty = mp.solutions.drawing_styles


def run_inference(source=0, use_onnx: bool = False, predict_every: int = 5):
    """
    Real-time sign language inference.

    Parameters
    ----------
    source       : int (webcam id) or str (video file path)
    use_onnx     : True -> ONNX Runtime (faster CPU)
                   False -> TensorFlow model
    predict_every: run model every N frames
    """
    if use_onnx:
        import onnxruntime as ort
        ort_sess = ort.InferenceSession(
            ONNX_FP32_PATH,
            providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
        print('Backend: ONNX Runtime')
    else:
        print('Backend: TensorFlow')

    cap          = cv2.VideoCapture(source)
    frame_buffer = deque(maxlen=SEQUENCE_LENGTH)
    tracker      = StabilizationTracker(window_size=15, majority_ratio=0.6, cooldown_s=1.0)
    frame_idx    = 0
    paused       = False
    last_text    = ''

    with mp_holistic.Holistic(
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5) as holistic:

        print(f'Started — source={source}')
        print('Controls: [q] quit | [r] reset sentence | [SPACE] pause')

        while cap.isOpened():
            if not paused:
                ret, bgr = cap.read()
                if not ret:
                    break

                rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
                rgb.flags.writeable = False
                results = holistic.process(rgb)
                rgb.flags.writeable = True
                bgr = cv2.cvtColor(rgb, cv2.COLOR_RGB2BGR)

                # Draw landmarks
                if results.pose_landmarks:
                    mp_drawing.draw_landmarks(
                        bgr, results.pose_landmarks,
                        mp_holistic.POSE_CONNECTIONS,
                        landmark_drawing_spec=mp_draw_sty.get_default_pose_landmarks_style())
                if results.left_hand_landmarks:
                    mp_drawing.draw_landmarks(
                        bgr, results.left_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
                if results.right_hand_landmarks:
                    mp_drawing.draw_landmarks(
                        bgr, results.right_hand_landmarks, mp_holistic.HAND_CONNECTIONS)
                if results.face_landmarks:
                    mp_drawing.draw_landmarks(
                        bgr, results.face_landmarks,
                        mp_holistic.FACEMESH_CONTOURS,
                        landmark_drawing_spec=None,
                        connection_drawing_spec=mp_draw_sty.get_default_face_mesh_contours_style())

                feat = mediapipe_to_232dim(results)
                frame_buffer.append(feat)
                frame_idx += 1

                # Run model every N frames
                if frame_idx % predict_every == 0 and len(frame_buffer) >= 10:
                    seq = np.array(frame_buffer, dtype=np.float32)
                    T   = len(seq)
                    X   = np.zeros((1, SEQUENCE_LENGTH, NUM_FEATURES), dtype=np.float32)
                    X[0, :T, :] = seq

                    if use_onnx:
                        logits = ort_sess.run(None, {'input': X})[0][0]
                    else:
                        logits = model_infer.predict(X, verbose=0)[0]

                    decoded, _ = tf.keras.backend.ctc_decode(
                        logits[np.newaxis], input_length=np.array([T]), greedy=True)
                    raw_text = decode_indices([x for x in decoded[0][0].numpy() if x != -1])

                    for w in raw_text.split():
                        tracker.update(w)
                    last_text = tracker.get_sentence()

            # Overlay text
            h_f, w_f = bgr.shape[:2]
            overlay = bgr.copy()
            cv2.rectangle(overlay, (0, h_f-80), (w_f, h_f), (0,0,0), -1)
            cv2.addWeighted(overlay, 0.5, bgr, 0.5, 0, bgr)
            cv2.putText(bgr, last_text or '(waiting…)',
                        (10, h_f-25), cv2.FONT_HERSHEY_SIMPLEX,
                        0.8, (0,255,100), 2)
            status = f'buf={len(frame_buffer)}/{SEQUENCE_LENGTH}  '\
                     f'{"PAUSED" if paused else "LIVE"}  '\
                     f'{"ONNX" if use_onnx else "TF"}'
            cv2.putText(bgr, status, (10,30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, (200,200,200), 1)

            cv2.imshow('CSLR — Sign Language Recognition', bgr)
            key = cv2.waitKey(1) & 0xFF
            if   key == ord('q'): break
            elif key == ord('r'):
                tracker.reset(); frame_buffer.clear(); last_text = ''
                print('Reset.')
            elif key == ord(' '):
                paused = not paused
                print('PAUSED' if paused else 'RESUMED')

    cap.release()
    cv2.destroyAllWindows()
    print('Final sentence:', tracker.get_sentence())


# Uncomment one of these to run:
# run_inference(source=0)                          # TF  + webcam
# run_inference(source=0,           use_onnx=True) # ONNX + webcam
# run_inference(source='video.mp4', use_onnx=False)# TF  + video file
print('run_inference() defined. Uncomment a call above to start.')


## 6. Offline Video File Evaluation

Processes any `.mp4` / `.avi` file in fixed-length chunks and prints timestamped predicted text.  
Useful for testing without a live webcam.

In [ ]:
# ============================================================
# 6. OFFLINE VIDEO EVALUATION
# ============================================================

def evaluate_video(video_path: str, chunk_frames: int = 150,
                   use_onnx: bool = False) -> list:
    if use_onnx:
        import onnxruntime as ort
        ort_sess = ort.InferenceSession(
            ONNX_FP32_PATH,
            providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])

    cap      = cv2.VideoCapture(video_path)
    fps      = cap.get(cv2.CAP_PROP_FPS) or 25
    n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    print(f'Video: {video_path}')
    print(f'  FPS={fps:.1f}  frames={n_frames}  '
          f'duration={n_frames/fps:.1f}s  chunk={chunk_frames}fr')

    results_list, buf, chunk_idx = [], [], 0

    with mp_holistic.Holistic(
            min_detection_confidence=0.5,
            min_tracking_confidence=0.5) as holistic:

        def _run_chunk(buf_frames, chunk_idx):
            seq = np.array(buf_frames, dtype=np.float32)
            T   = len(seq)
            X   = np.zeros((1, SEQUENCE_LENGTH, NUM_FEATURES), dtype=np.float32)
            X[0, :T, :] = seq
            if use_onnx:
                logits = ort_sess.run(None, {'input': X})[0][0]
            else:
                logits = model_infer.predict(X, verbose=0)[0]
            decoded, _ = tf.keras.backend.ctc_decode(
                logits[np.newaxis], input_length=np.array([T]), greedy=True)
            text = decode_indices([x for x in decoded[0][0].numpy() if x != -1])
            t_s  = chunk_idx * chunk_frames / fps
            t_e  = t_s + len(buf_frames) / fps
            print(f'  [{t_s:6.1f}s – {t_e:6.1f}s]  {text or "(silence)"}')
            return {'start': t_s, 'end': t_e, 'text': text}

        while cap.isOpened():
            ret, bgr = cap.read()
            if not ret:
                break
            rgb     = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
            results = holistic.process(rgb)
            buf.append(mediapipe_to_232dim(results))

            if len(buf) == chunk_frames:
                results_list.append(_run_chunk(buf, chunk_idx))
                buf = []; chunk_idx += 1

        if buf:
            results_list.append(_run_chunk(buf, chunk_idx))

    cap.release()
    print(f'Done. {len(results_list)} chunks processed.')
    return results_list


# Example:
# predictions = evaluate_video('my_signing_video.mp4', chunk_frames=150)
print('evaluate_video() is ready. Pass a video path to run it.')
